# Scientific visualization and simulation

Turn numerical experiments into honest, interpretable, reproducible figures with Matplotlib,
Seaborn, pandas, and Plotly.

**Lecture 5 · Notebook 00 · CMOR 438 / INDE 577**

## Orientation: a visualization studio, not a chart catalog

**Live core:** visual questions and encodings, Matplotlib's object model, relational and distribution
plots, uncertainty, statistical graphics with Seaborn, accessibility, and reproducible export.

**Practice:** critique a misleading figure, design a distribution comparison, and build a reusable
plotting function with tests.

**Extension:** interactive Plotly figures, random walks, Monte Carlo convergence, nonlinear dynamics,
animation choices, and large-data strategies.

This is deliberately a long reference. A three-hour meeting should select a core route; the remaining
examples become guided study and a visual toolbox for later projects.

## How to use this notebook

**Estimated time:** 165 minutes core, plus 120 minutes of practice and extension.

**Prerequisites:** NumPy arrays, pandas tables, functions, tests, and the Rice DSM environment. Run
`uv sync`, choose the **Rice DSM** kernel in VS Code, then restart and run all.

For every figure, write the question first. Before running its cell, predict the marks, encodings,
scales, aggregation, and uncertainty. Afterward, state one supported observation and one conclusion
the figure cannot establish. Do not confuse visual salience with scientific importance.

## Learning objectives

By the end, you should be able to:

- translate an analytical question into appropriate marks, encodings, scales, and facets;
- explain Matplotlib's `Figure`–`Axes`–`Axis`–`Artist` hierarchy;
- build labeled line, scatter, distribution, categorical, matrix, and small-multiple plots;
- distinguish observations, estimates, variability, standard error, and confidence intervals;
- use Seaborn's axes-level and figure-level APIs with tidy pandas data;
- use color, position, shape, and line style accessibly and intentionally;
- diagnose overplotting, bin sensitivity, truncated axes, inappropriate smoothing, and hidden aggregation;
- create deterministic simulations without mistaking a seed for scientific robustness;
- build an interactive Plotly figure and explain when interactivity helps or harms communication;
- save publication, web, and review artifacts with explicit size, format, and provenance; and
- test plotting functions without asserting fragile implementation details.

## Why this matters

Visualization is used to discover defects, understand distributions, communicate model behavior,
monitor systems, and support decisions. It is also a statistical transformation: binning, smoothing,
aggregation, axis limits, normalization, and color mapping change what becomes visible.

A polished chart can amplify a bad comparison. A truthful chart can still be inaccessible. A useful
exploratory plot may be too dense for an executive explanation. Professional visualization starts
with purpose and audience, then makes the transformation from data to pixels inspectable.

## Worked examples: scientific studios

We use several small, deterministic simulations:

1. **Heat diffusion:** repeated temperature measurements support lines, uncertainty, facets, and
   residual diagnostics.
2. **Probability:** mixtures and sampling distributions expose histograms, ECDFs, KDEs, and box or
   violin plots.
3. **Random walks and Monte Carlo:** trajectories and convergence show why simulation output needs
   ensembles and uncertainty.
4. **Nonlinear dynamics:** the logistic map produces trajectories and a bifurcation diagram.

The simulations are teaching models, not empirical evidence about physical materials or real systems.

## Professional practice

| Data scientist asks | Software engineer asks |
| --- | --- |
| Which comparison answers the scientific question? | Which function and data contract produce it? |
| What was filtered, aggregated, or transformed? | Is that transformation explicit and tested? |
| What uncertainty or denominator is shown? | Can the computation be reproduced independently of rendering? |
| Does the encoding exaggerate or hide variation? | Are limits, scales, palettes, and defaults controlled? |
| Can all viewers distinguish the groups? | Are labels, contrast, redundant encodings, and text alternatives present? |
| Is this exploratory evidence or a final claim? | Which artifact, metadata, and version are reviewed? |

The objective is not “make it pretty.” The objective is make the intended comparison easy and the
limitations hard to miss.

## 1. Confirm the visualization environment

Matplotlib supplies low-level control and a reusable object model. Seaborn builds statistical,
dataset-oriented graphics on Matplotlib. pandas offers convenient plotting methods. Plotly creates
interactive figure specifications rendered in a browser or notebook.

These packages are declared in `pyproject.toml` and locked in `uv.lock`; no cell installs packages.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import seaborn as sns
from matplotlib.axes import Axes
from matplotlib.figure import Figure

print("Matplotlib:", mpl.__version__)
print("Seaborn:   ", sns.__version__)
print("Plotly:    ", plotly.__version__)

assert int(mpl.__version__.split(".")[0]) >= 3
assert int(sns.__version__.split(".")[0]) >= 0
assert int(plotly.__version__.split(".")[0]) >= 7

The notebook kernel supplies an inline renderer. Scripts and CI may use a noninteractive backend such
as Agg. A backend is rendering infrastructure, not the visual design itself. Saving through the
`Figure` object works across Windows, macOS, and Linux when paths use `pathlib`.

In [ ]:
sns.set_theme(
    context="notebook",
    style="whitegrid",
    palette="colorblind",
)

COURSE_PALETTE = sns.color_palette("colorblind")
assert len(COURSE_PALETTE) >= 6

## 2. Begin with a visual question

A useful planning chain is:

```text
question → analytical unit → data transformation → marks
         → visual encodings → scales → guides/facets → annotation → artifact
```

- **Marks** are geometric objects: points, lines, bars, areas, text, or cells.
- **Encodings** map variables to position, length, color, size, shape, opacity, or line style.
- **Scales** map data domains to visual ranges and tick positions.
- **Guides**—axes, legends, and colorbars—explain those mappings.
- **Facets** repeat a common design across subsets, preferably with comparable scales.

Position on a common scale usually supports more precise comparison than area, angle, or volume.

### Four visualization modes

| Mode | Primary purpose | Typical behavior |
| --- | --- | --- |
| exploratory | discover structure and defects | many quick views; dense context |
| diagnostic | assess a model or pipeline | residuals, calibration, drift, missingness |
| explanatory | support a focused claim | edited comparison, annotation, restrained detail |
| operational | monitor a changing system | stable definitions, thresholds, ownership, alert context |

The same dataset may need different figures for each mode.

## 3. Simulate a tidy heat-diffusion experiment

For material \(m\), distance \(d\), and time \(t\), the teaching signal is

\[
T(t,d,m) = T_{ambient} + A_m e^{-t/70}e^{-d/14}.
\]

We add independent Gaussian measurement noise. The formula is intentionally simple; it is not a
validated heat-equation solution for copper or aluminum.

In [ ]:
def simulate_heat_diffusion(
    *,
    seed: int,
    replicates: int = 12,
) -> pd.DataFrame:
    """Simulate repeated temperature measurements for plotting lessons.

    Parameters
    ----------
    seed : int
        Seed for a local NumPy random generator.
    replicates : int, default=12
        Positive number of independent measurements at each condition.

    Returns
    -------
    pandas.DataFrame
        Tidy table with one row per material–time–sensor–replicate
        observation. Temperatures are measured in degrees Celsius.

    Raises
    ------
    ValueError
        If ``replicates`` is not positive.

    Notes
    -----
    This deterministic teaching simulation is not empirical material data.
    """

    if replicates <= 0:
        raise ValueError("replicates must be positive")

    simulation_rng = np.random.default_rng(seed)
    materials = {"copper": 62.0, "aluminum": 48.0}
    times_s = np.arange(0, 121, 15, dtype=np.int64)
    sensors = {"S1": 1.0, "S2": 2.5, "S3": 5.0, "S4": 8.0}
    ambient_c = 22.0
    records: list[dict[str, object]] = []

    for material, amplitude_c in materials.items():
        for time_s in times_s:
            for sensor_id, distance_cm in sensors.items():
                expected_c = ambient_c + (
                    amplitude_c
                    * np.exp(-time_s / 70.0)
                    * np.exp(-distance_cm / 14.0)
                )
                for replicate in range(replicates):
                    records.append(
                        {
                            "material": material,
                            "time_s": int(time_s),
                            "sensor_id": sensor_id,
                            "distance_cm": distance_cm,
                            "replicate": replicate,
                            "expected_temperature_c": expected_c,
                            "temperature_c": expected_c
                            + simulation_rng.normal(loc=0.0, scale=1.25),
                        }
                    )

    return pd.DataFrame.from_records(records).astype(
        {
            "material": "category",
            "sensor_id": "string",
            "time_s": "int64",
            "replicate": "int64",
        }
    )


heat_data = simulate_heat_diffusion(seed=438, replicates=12)

assert heat_data.shape == (864, 7)
assert not heat_data.isna().any().any()
assert not heat_data.duplicated(
    subset=["material", "time_s", "sensor_id", "replicate"]
).any()
heat_data.head()

The table is long/tidy: each row is an observation, each column is a variable, and each observational
unit has a unique key. This form works naturally with Seaborn and Plotly semantic mappings.

## 4. Matplotlib's explicit object model

```text
Figure                     whole output canvas
└── Axes                   one plotting region (despite the plural name)
    ├── xaxis and yaxis    scales, ticks, tick labels
    └── Artists            lines, points, text, legends, patches, images
```

Prefer the explicit object-oriented style for reusable or multi-panel figures:
`figure, axis = plt.subplots()` followed by `axis.plot(...)`. The stateful `plt.plot(...)` style is
convenient for quick exploration but can make ownership ambiguous in larger code.

In [ ]:
time_grid_s = np.linspace(0.0, 120.0, num=200)
modeled_temperature_c = 22.0 + 62.0 * np.exp(-time_grid_s / 70.0)

figure, axis = plt.subplots(figsize=(7.5, 4.5), layout="constrained")
line_artists = axis.plot(
    time_grid_s,
    modeled_temperature_c,
    color=COURSE_PALETTE[0],
    linewidth=2.5,
    label="model",
)
axis.set(
    title="Modeled cooling at the heat source",
    xlabel="Time (s)",
    ylabel="Temperature (°C)",
)
axis.legend(frameon=False)

assert isinstance(figure, Figure)
assert isinstance(axis, Axes)
assert len(line_artists) == 1
assert axis.get_xlabel() == "Time (s)"
display(figure)
plt.close(figure)

A line connects ordered values and implies continuity or sequence. Do not connect independent
categories or observations when the segments imply transitions that do not exist.

### Plot raw observations before their summary

Summary lines can hide sample size, outliers, multimodality, and unequal variation. Begin by seeing
the observations for one focused condition.

In [ ]:
copper_s1 = heat_data.loc[
    (heat_data["material"] == "copper")
    & (heat_data["sensor_id"] == "S1")
]

figure, axis = plt.subplots(figsize=(7.5, 4.5), layout="constrained")
axis.scatter(
    copper_s1["time_s"],
    copper_s1["temperature_c"],
    alpha=0.35,
    s=28,
    color=COURSE_PALETTE[0],
    edgecolors="none",
    label="replicate observations",
)
axis.plot(
    copper_s1["time_s"],
    copper_s1["expected_temperature_c"],
    color="black",
    linewidth=2,
    label="simulation expectation",
)
axis.set(
    title="Repeated copper measurements at sensor S1",
    xlabel="Time (s)",
    ylabel="Temperature (°C)",
)
axis.legend(frameon=False)

assert len(axis.collections) == 1
assert len(axis.lines) == 1
display(figure)
plt.close(figure)

Opacity reduces overplotting but changes perceived density and behaves differently on screens and
print. Other responses include smaller marks, jitter for discrete positions, hexagonal binning,
contours, sampling with a declared policy, aggregation, or faceting.

### Multiple series need semantic and accessible distinction

Color can encode material, while line style redundantly encodes the same groups. Redundancy helps
viewers with color-vision differences and grayscale output.

In [ ]:
profile_summary = (
    heat_data.loc[heat_data["sensor_id"] == "S2"]
    .groupby(["material", "time_s"], observed=True)
    .agg(
        mean_temperature_c=("temperature_c", "mean"),
        standard_error_c=(
            "temperature_c",
            lambda values: values.std(ddof=1) / np.sqrt(values.count()),
        ),
        observations=("temperature_c", "size"),
    )
    .reset_index()
)

figure, axis = plt.subplots(figsize=(8, 4.8), layout="constrained")
style_by_material = {"copper": "-", "aluminum": "--"}
for material, group in profile_summary.groupby("material", observed=True):
    axis.plot(
        group["time_s"],
        group["mean_temperature_c"],
        marker="o",
        linestyle=style_by_material[str(material)],
        linewidth=2,
        label=str(material).title(),
    )

axis.set(
    title="Mean cooling profiles at sensor S2",
    xlabel="Time (s)",
    ylabel="Mean temperature (°C)",
)
axis.legend(title="Material", frameon=False)

assert len(axis.lines) == 2
display(figure)
plt.close(figure)

## 5. Small multiples often beat overloaded plots

Facets repeat a visual structure across groups. Shared scales enable comparison; independent scales
may reveal within-group shape but conceal magnitude differences. State which comparison matters.

In [ ]:
figure, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(10, 4.2),
    sharex=True,
    sharey=True,
    layout="constrained",
)

for axis, material in zip(axes, ["copper", "aluminum"], strict=True):
    material_data = heat_data.loc[heat_data["material"] == material]
    for sensor_id, group in material_data.groupby("sensor_id"):
        sensor_means = group.groupby("time_s")["temperature_c"].mean()
        axis.plot(
            sensor_means.index,
            sensor_means,
            marker="o",
            label=str(sensor_id),
        )
    axis.set_title(material.title())
    axis.set_xlabel("Time (s)")

axes[0].set_ylabel("Mean temperature (°C)")
axes[1].legend(title="Sensor", frameon=False)
figure.suptitle("Cooling by material and sensor distance")

assert axes.shape == (2,)
assert all(len(axis.lines) == 4 for axis in axes)
display(figure)
plt.close(figure)

### `subplot_mosaic` gives semantic panel names

Named layouts are easier to maintain than remembering that `axes[1, 0]` is the residual panel. Use a
mosaic when panels have different roles or spans.

In [ ]:
mosaic_figure, mosaic_axes = plt.subplot_mosaic(
    [["profile", "profile"], ["residual", "distribution"]],
    figsize=(10, 7),
    layout="constrained",
)

mosaic_axes["profile"].plot(time_grid_s, modeled_temperature_c)
mosaic_axes["profile"].set_title("Profile")
mosaic_axes["residual"].axhline(0.0, color="black", linewidth=1)
mosaic_axes["residual"].set_title("Residual reference")
mosaic_axes["distribution"].hist(copper_s1["temperature_c"], bins=12)
mosaic_axes["distribution"].set_title("Observed distribution")

assert set(mosaic_axes) == {"profile", "residual", "distribution"}
display(mosaic_figure)
plt.close(mosaic_figure)

## 6. Scales are part of the claim

A logarithmic scale represents equal ratios with equal visual distance. It can reveal multiplicative
structure over several orders of magnitude. Zero and negative values are outside a standard log
domain. Never use a log scale merely to make a trend look straighter without explaining it.

In [ ]:
algorithm_sizes = np.logspace(1, 6, num=30)
linear_work = algorithm_sizes
quadratic_work = algorithm_sizes**2

figure, axes = plt.subplots(1, 2, figsize=(10, 4.2), layout="constrained")
axes[0].plot(algorithm_sizes, linear_work, label=r"$n$")
axes[0].plot(algorithm_sizes, quadratic_work, label=r"$n^2$")
axes[0].set(title="Linear axes", xlabel="Problem size n", ylabel="Relative work")

axes[1].loglog(algorithm_sizes, linear_work, label=r"$n$")
axes[1].loglog(algorithm_sizes, quadratic_work, label=r"$n^2$")
axes[1].set(title="Log–log axes", xlabel="Problem size n", ylabel="Relative work")
axes[1].legend(frameon=False)

assert axes[1].get_xscale() == "log"
assert axes[1].get_yscale() == "log"
display(figure)
plt.close(figure)

For bars, length normally encodes magnitude from a meaningful baseline, so a truncated quantitative
axis can grossly exaggerate differences. Lines and scatter plots do not always require zero because
position rather than bar length carries the comparison. The choice depends on the encoding.

## 7. Uncertainty must name the quantity

Raw observation spread, standard deviation, standard error, confidence intervals, prediction
intervals, and posterior intervals answer different questions. A shaded ribbon is not self-explanatory.

For independent observations, a common descriptive standard error is
\(SE = s/\sqrt{n}\). The interval `mean ± 1.96 × SE` is an approximate 95% confidence interval under
specific assumptions; it is not a range containing 95% of individual temperatures.

In [ ]:
profile_summary = profile_summary.assign(
    ci95_low_c=lambda frame: (
        frame["mean_temperature_c"] - 1.96 * frame["standard_error_c"]
    ),
    ci95_high_c=lambda frame: (
        frame["mean_temperature_c"] + 1.96 * frame["standard_error_c"]
    ),
)

figure, axis = plt.subplots(figsize=(8, 4.8), layout="constrained")
for color, (material, group) in zip(
    COURSE_PALETTE,
    profile_summary.groupby("material", observed=True),
    strict=False,
):
    axis.plot(
        group["time_s"],
        group["mean_temperature_c"],
        color=color,
        linewidth=2,
        label=str(material).title(),
    )
    axis.fill_between(
        group["time_s"],
        group["ci95_low_c"],
        group["ci95_high_c"],
        color=color,
        alpha=0.2,
        linewidth=0,
    )

axis.set(
    title="Mean temperature with approximate 95% confidence intervals",
    xlabel="Time (s)",
    ylabel="Temperature (°C)",
)
axis.legend(title="Material", frameon=False)

assert len(axis.lines) == 2
assert len(axis.collections) == 2
display(figure)
plt.close(figure)

Repeated measurements may share instruments or experimental runs and therefore violate independence.
In real work, the unit of replication and dependence structure determine the uncertainty calculation.
Plotting software cannot decide that design question.

### Error bars suit isolated estimates

Whiskers are often clearer than bands for a small number of separate estimates. State whether the
whisker is a standard deviation, standard error, confidence interval, or another quantity.

In [ ]:
final_time_summary = (
    heat_data.loc[heat_data["time_s"] == 120]
    .groupby(["material", "sensor_id"], observed=True)
    .agg(
        mean_c=("temperature_c", "mean"),
        standard_deviation_c=("temperature_c", "std"),
    )
    .reset_index()
)

figure, axis = plt.subplots(figsize=(8, 4.8), layout="constrained")
for position, material in enumerate(["copper", "aluminum"]):
    group = final_time_summary.loc[final_time_summary["material"] == material]
    axis.errorbar(
        np.arange(len(group)) + 0.08 * (position - 0.5),
        group["mean_c"],
        yerr=group["standard_deviation_c"],
        fmt="o",
        capsize=4,
        label=f"{material.title()} (mean ± SD)",
    )

axis.set_xticks(np.arange(4), ["S1", "S2", "S3", "S4"])
axis.set(
    title="Temperature at 120 seconds",
    xlabel="Sensor",
    ylabel="Temperature (°C)",
)
axis.legend(frameon=False)

assert len(axis.lines) >= 2
display(figure)
plt.close(figure)

## 8. Distribution plots answer different questions

No single distribution plot is universally best:

- a **histogram** shows binned counts or density and depends on bin edges;
- an **ECDF** shows the fraction of observations at or below every observed value without bins;
- a **KDE** shows a smoothed density estimate and depends on kernel/bandwidth and boundary behavior;
- a **box plot** compresses median, quartiles, and a convention for whiskers/outliers;
- a **violin plot** mirrors a KDE and can hide sample size or unsupported density;
- a **strip/swarm plot** preserves individual observations but can overplot.

### Histogram bin choices can change the story

We simulate a mixture distribution: most instrument noise is narrow, with a small shifted component.
Compare multiple reasonable bin counts rather than accepting one default as truth.

In [ ]:
distribution_rng = np.random.default_rng(seed=577)
noise_component = distribution_rng.normal(0.0, 1.0, size=1_800)
shifted_component = distribution_rng.normal(3.5, 0.55, size=200)
mixture_samples = np.concatenate([noise_component, shifted_component])

figure, axes = plt.subplots(
    1,
    3,
    figsize=(12, 3.8),
    sharex=True,
    sharey=True,
    layout="constrained",
)
for axis, bin_count in zip(axes, [10, 30, 80], strict=True):
    axis.hist(
        mixture_samples,
        bins=bin_count,
        density=True,
        color=COURSE_PALETTE[0],
        alpha=0.8,
    )
    axis.set_title(f"{bin_count} bins")
    axis.set_xlabel("Measurement residual (°C)")
axes[0].set_ylabel("Estimated density")
figure.suptitle("One sample, three histogram resolutions")

assert mixture_samples.shape == (2_000,)
display(figure)
plt.close(figure)

Density normalization makes total bar area one; the y-axis is density, not probability per bar and
not observation count. Report bin policy for reviewable analyses.

### ECDF preserves every observed threshold

The empirical cumulative distribution function is monotone from roughly `1/n` to 1. It makes medians,
quantiles, and stochastic comparisons visible without a bandwidth.

In [ ]:
sorted_samples = np.sort(mixture_samples)
cumulative_fraction = np.arange(1, sorted_samples.size + 1) / sorted_samples.size

figure, axis = plt.subplots(figsize=(7.5, 4.5), layout="constrained")
axis.step(
    sorted_samples,
    cumulative_fraction,
    where="post",
    color=COURSE_PALETTE[1],
)
axis.axhline(0.5, color="black", linestyle="--", linewidth=1)
axis.set(
    title="Empirical cumulative distribution of residuals",
    xlabel="Measurement residual (°C)",
    ylabel="Fraction ≤ residual",
    ylim=(0.0, 1.0),
)

assert np.all(np.diff(cumulative_fraction) > 0)
display(figure)
plt.close(figure)

## 9. Seaborn maps tidy variables to visual semantics

Seaborn integrates with pandas and uses names such as `x`, `y`, `hue`, `style`, `size`, `row`, and
`col` to describe mappings. **Axes-level** functions such as `sns.scatterplot` draw on a supplied
Matplotlib `Axes`. **Figure-level** functions such as `sns.relplot`, `sns.displot`, and `sns.catplot`
manage an entire grid and return a grid object.

### Relational plot: observation-level scatter with redundant encodings

Color and marker style both represent material; size represents sensor distance. Too many encodings
can overload a plot, so use this for exploration and simplify for explanation.

In [ ]:
sampled_heat_data = heat_data.sample(n=220, random_state=438)

figure, axis = plt.subplots(figsize=(8, 5), layout="constrained")
sns.scatterplot(
    data=sampled_heat_data,
    x="time_s",
    y="temperature_c",
    hue="material",
    style="material",
    size="distance_cm",
    sizes=(25, 110),
    alpha=0.65,
    ax=axis,
)
axis.set(
    title="Sampled heat-diffusion observations",
    xlabel="Time (s)",
    ylabel="Temperature (°C)",
)
sns.move_legend(axis, "upper right", frameon=False)

assert len(axis.collections) >= 1
display(figure)
plt.close(figure)

The sampling seed makes the displayed subset reproducible. It does not guarantee the subset preserves
rare groups. For high-stakes communication, define stratification and compare the sample with the
full data.

### Figure-level facets: compare materials and sensors

`relplot` constructs a `FacetGrid`. Here each panel has the same axes so slopes and magnitudes remain
comparable. Seaborn estimates means and uncertainty from repeated observations; we explicitly request
standard-deviation error bands rather than leaving their meaning implicit.

In [ ]:
relationship_grid = sns.relplot(
    data=heat_data,
    kind="line",
    x="time_s",
    y="temperature_c",
    hue="material",
    style="material",
    col="sensor_id",
    col_wrap=2,
    estimator="mean",
    errorbar="sd",
    markers=True,
    facet_kws={"sharex": True, "sharey": True},
    height=3.1,
    aspect=1.2,
)
relationship_grid.set_axis_labels("Time (s)", "Temperature (°C)")
relationship_grid.set_titles("Sensor {col_name}")
relationship_grid.figure.suptitle(
    "Mean cooling profiles with ±1 SD bands",
    y=1.02,
)

assert relationship_grid.axes.size == 4
display(relationship_grid.figure)
plt.close(relationship_grid.figure)

Seaborn's statistical defaults are useful for exploration but must be reviewed. Know the estimator,
uncertainty method, bootstrap behavior, grouping unit, and missing-value policy before treating the
figure as inferential evidence.

### KDE and ECDF side by side

Kernel density is smooth but may place density outside a physically possible range. `cut=0` limits
display beyond observed extremes; it does not solve boundary bias. ECDF is less visually smooth but
closer to the observed sample.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(10, 4.3), layout="constrained")
sns.kdeplot(
    data=copper_s1,
    x="temperature_c",
    hue="time_s",
    palette="viridis",
    common_norm=False,
    cut=0,
    ax=axes[0],
)
axes[0].set(title="KDE by time", xlabel="Temperature (°C)", ylabel="Density")

sns.ecdfplot(
    data=copper_s1,
    x="temperature_c",
    hue="time_s",
    palette="viridis",
    ax=axes[1],
)
axes[1].set(
    title="ECDF by time",
    xlabel="Temperature (°C)",
    ylabel="Cumulative fraction",
)

assert len(axes[0].lines) > 0
assert len(axes[1].lines) > 0
display(figure)
plt.close(figure)

### Categorical distributions: show observations and summaries together

A box plot summarizes quartiles, while jittered points retain sample size and unusual observations.
The category order is analytical, not alphabetical decoration.

In [ ]:
late_heat_data = heat_data.loc[heat_data["time_s"] == 120]

figure, axis = plt.subplots(figsize=(8.5, 4.8), layout="constrained")
sns.boxplot(
    data=late_heat_data,
    x="sensor_id",
    y="temperature_c",
    hue="material",
    whis=1.5,
    showfliers=False,
    ax=axis,
)
sns.stripplot(
    data=late_heat_data,
    x="sensor_id",
    y="temperature_c",
    hue="material",
    dodge=True,
    alpha=0.45,
    size=3,
    color="black",
    legend=False,
    ax=axis,
)
axis.set(
    title="Final-time distributions: summary plus observations",
    xlabel="Sensor",
    ylabel="Temperature (°C)",
)
sns.move_legend(axis, "upper right", title="Material", frameon=False)

assert len(axis.patches) > 0
assert len(axis.collections) > 0
display(figure)
plt.close(figure)

Box-plot “outliers” are values beyond a graphical whisker convention, not automatically bad data.
Never delete an observation because a plotting function marked it as a flier.

### Violin plots reveal shape but depend on smoothing

Split violins can compare two groups compactly. With small samples they may imply more distributional
resolution than the data support, so pair them with counts or raw observations.

In [ ]:
figure, axis = plt.subplots(figsize=(8.5, 4.8), layout="constrained")
sns.violinplot(
    data=late_heat_data,
    x="sensor_id",
    y="temperature_c",
    hue="material",
    split=True,
    inner="quart",
    cut=0,
    density_norm="width",
    ax=axis,
)
axis.set(
    title="Smoothed final-time distributions",
    xlabel="Sensor",
    ylabel="Temperature (°C)",
)
sns.move_legend(axis, "upper right", title="Material", frameon=False)

display(figure)
plt.close(figure)

## 10. Regression graphics are exploratory guides

A fitted line in a scatter plot does not establish causality, an adequate model, independent errors,
or calibrated uncertainty. Always inspect residuals and the data-generating design.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(10.5, 4.3), layout="constrained")
sns.regplot(
    data=sampled_heat_data,
    x="distance_cm",
    y="temperature_c",
    scatter_kws={"alpha": 0.25, "s": 24},
    line_kws={"color": COURSE_PALETTE[3]},
    ax=axes[0],
)
axes[0].set(
    title="A misleading pooled linear guide",
    xlabel="Sensor distance (cm)",
    ylabel="Temperature (°C)",
)

sns.scatterplot(
    data=sampled_heat_data,
    x="expected_temperature_c",
    y="temperature_c",
    hue="material",
    alpha=0.55,
    ax=axes[1],
)
comparison_limits = [20.0, 85.0]
axes[1].plot(comparison_limits, comparison_limits, color="black", linestyle="--")
axes[1].set(
    title="Observed versus expected",
    xlabel="Expected temperature (°C)",
    ylabel="Observed temperature (°C)",
)
sns.move_legend(axes[1], "upper left", frameon=False)

display(figure)
plt.close(figure)

The pooled left panel ignores time, material, repeated observations, and the nonlinear model. Its line
is easy to draw and hard to interpret. Visualization should expose design structure rather than erase
it for a cleaner trend.

### Residual plots test visual expectations

Residuals should be defined explicitly. Here `observed − simulation expectation` isolates the added
noise because the expectation is known by construction.

In [ ]:
diagnostic_data = heat_data.assign(
    residual_c=lambda frame: (
        frame["temperature_c"] - frame["expected_temperature_c"]
    )
)

figure, axes = plt.subplots(1, 2, figsize=(10.5, 4.3), layout="constrained")
sns.scatterplot(
    data=diagnostic_data.sample(300, random_state=577),
    x="expected_temperature_c",
    y="residual_c",
    hue="material",
    alpha=0.5,
    ax=axes[0],
)
axes[0].axhline(0.0, color="black", linewidth=1)
axes[0].set(
    title="Residuals versus expected value",
    xlabel="Expected temperature (°C)",
    ylabel="Residual (°C)",
)

sns.histplot(
    data=diagnostic_data,
    x="residual_c",
    hue="material",
    element="step",
    stat="density",
    common_norm=False,
    kde=True,
    ax=axes[1],
)
axes[1].set(
    title="Residual distributions",
    xlabel="Residual (°C)",
    ylabel="Density",
)

assert abs(diagnostic_data["residual_c"].mean()) < 0.2
display(figure)
plt.close(figure)

## 11. Matrix displays and colormaps

Heatmaps encode values with color over a two-dimensional grid. Use a **sequential** colormap for
ordered magnitudes, a **diverging** colormap around a meaningful center such as zero, and a
**qualitative** palette for unordered categories. Rainbow maps often introduce false boundaries and
nonuniform perceptual emphasis.

In [ ]:
mean_temperature_matrix = (
    heat_data.loc[heat_data["material"] == "copper"]
    .groupby(["time_s", "sensor_id"])["temperature_c"]
    .mean()
    .unstack("sensor_id")
)

figure, axis = plt.subplots(figsize=(7, 5), layout="constrained")
sns.heatmap(
    mean_temperature_matrix,
    cmap="mako",
    annot=True,
    fmt=".1f",
    cbar_kws={"label": "Mean temperature (°C)"},
    ax=axis,
)
axis.set(
    title="Copper temperature across time and sensors",
    xlabel="Sensor",
    ylabel="Time (s)",
)

assert mean_temperature_matrix.shape == (9, 4)
assert len(figure.axes) == 2  # plot plus colorbar
display(figure)
plt.close(figure)

Cell annotation helps exact lookup in a small matrix; it becomes clutter in a large one. A colorbar
must name the quantity and unit. Do not interpret color differences more precisely than the scale and
measurement warrant.

### Correlation heatmaps need caution

Correlation is pairwise association, not causality or feature importance. Repeated observations,
nonlinearity, outliers, group structure, and leakage can make a clean matrix misleading.

In [ ]:
numeric_columns = [
    "time_s",
    "distance_cm",
    "expected_temperature_c",
    "temperature_c",
    "residual_c",
]
correlation_matrix = diagnostic_data[numeric_columns].corr()

figure, axis = plt.subplots(figsize=(7, 5.5), layout="constrained")
sns.heatmap(
    correlation_matrix,
    cmap="vlag",
    center=0.0,
    vmin=-1.0,
    vmax=1.0,
    annot=True,
    fmt=".2f",
    square=True,
    cbar_kws={"label": "Pearson correlation"},
    ax=axis,
)
axis.set_title("Pairwise correlations in the teaching simulation")

assert np.allclose(np.diag(correlation_matrix), 1.0)
display(figure)
plt.close(figure)

## 12. pandas plotting is convenient, not a separate rendering engine

`Series.plot` and `DataFrame.plot` normally create Matplotlib artists. Pass an `Axes` so the plot joins
a controlled composition rather than relying on hidden current state.

In [ ]:
pandas_plot_data = mean_temperature_matrix.rename(
    columns=lambda sensor: f"Sensor {sensor}"
)

figure, axis = plt.subplots(figsize=(8, 4.8), layout="constrained")
pandas_plot_data.plot(marker="o", ax=axis)
axis.set(
    title="pandas convenience plotting on a Matplotlib Axes",
    xlabel="Time (s)",
    ylabel="Mean temperature (°C)",
)
axis.legend(title="Series", frameon=False)

assert len(axis.lines) == 4
display(figure)
plt.close(figure)

## 13. Annotations should carry analytical meaning

Use text sparingly to identify an event, threshold, optimum, failure, or takeaway. Annotation is not a
substitute for labeled axes and a clear design.

In [ ]:
copper_s1_means = (
    copper_s1.groupby("time_s")["temperature_c"].mean().sort_index()
)
threshold_c = 40.0
crossing_times = copper_s1_means.index[copper_s1_means < threshold_c]
first_crossing_s = int(crossing_times.min())

figure, axis = plt.subplots(figsize=(8, 4.5), layout="constrained")
axis.plot(copper_s1_means.index, copper_s1_means, marker="o")
axis.axhline(
    threshold_c,
    color=COURSE_PALETTE[3],
    linestyle="--",
    label="40 °C reference",
)
axis.annotate(
    f"first sampled mean below 40 °C: {first_crossing_s} s",
    xy=(first_crossing_s, copper_s1_means.loc[first_crossing_s]),
    xytext=(55, 52),
    arrowprops={"arrowstyle": "->", "color": "black"},
)
axis.set(
    title="A domain-relevant threshold annotation",
    xlabel="Time (s)",
    ylabel="Mean temperature (°C)",
)
axis.legend(frameon=False)

display(figure)
plt.close(figure)

The phrase “first sampled mean” matters: the experiment observes a discrete time grid and noisy
replicates. The plot does not establish the exact continuous crossing time.

## 14. Accessibility is part of correctness

At minimum:

- write a takeaway-oriented title when communicating a result;
- label axes with quantities and units;
- do not encode a critical distinction with color alone;
- use palettes with adequate contrast and an appropriate luminance structure;
- keep text legible at final display size;
- avoid dense legends when direct labels or facets are clearer;
- provide a concise text alternative describing structure and key comparison; and
- verify grayscale, projector, dark/light background, and color-vision conditions relevant to the audience.

“Colorblind-friendly” is not a guarantee for every viewer or display.

In [ ]:
accessibility_figure, accessibility_axis = plt.subplots(
    figsize=(8, 4.5),
    layout="constrained",
)
for color, marker, linestyle, (material, group) in zip(
    COURSE_PALETTE,
    ["o", "s"],
    ["-", "--"],
    profile_summary.groupby("material", observed=True),
    strict=False,
):
    accessibility_axis.plot(
        group["time_s"],
        group["mean_temperature_c"],
        color=color,
        marker=marker,
        linestyle=linestyle,
        linewidth=2.2,
        label=str(material).title(),
    )

accessibility_axis.set(
    title="Copper remains warmer than aluminum in this simulation",
    xlabel="Time since heating began (s)",
    ylabel="Mean sensor temperature (°C)",
)
accessibility_axis.legend(title="Material", frameon=False)

assert accessibility_axis.get_title()
assert accessibility_axis.get_xlabel().endswith("(s)")
assert accessibility_axis.get_ylabel().endswith("(°C)")
display(accessibility_figure)
plt.close(accessibility_figure)

**Text alternative:** “At sensor S2, mean simulated temperature decreases over time for both
materials. Copper is warmer than aluminum at every sampled time. Bands shown in the earlier figure
represent approximate confidence intervals for the simulated mean, not individual prediction ranges.”

## 15. Common ways to mislead—even accidentally

1. truncate a bar axis so a small difference appears enormous;
2. hide observations behind an aggregate or smooth curve;
3. use area or volume when position would support comparison;
4. choose bins or bandwidth after seeing which tells the preferred story;
5. encode ordered magnitude with an unordered or nonuniform palette;
6. compare groups with different denominators but omit counts;
7. use dual y-axes whose scales can manufacture almost any apparent alignment;
8. connect unordered categories with lines;
9. omit uncertainty or display it without definition;
10. cherry-pick time windows, subgroups, or axis limits;
11. treat model output as observed truth; or
12. show an interactive default state that hides the important failure subgroup.

Intent does not determine effect. Review figures as analytical code.

### A truncated bar-axis demonstration

Both panels contain identical means. The right panel's narrow y-range makes a two-degree difference
look dominant. This controlled comparison is for critique, not a template.

In [ ]:
comparison_means = pd.Series(
    {"Method A": 82.0, "Method B": 84.0},
    name="accuracy_percent",
)

figure, axes = plt.subplots(1, 2, figsize=(9.5, 4), layout="constrained")
for axis in axes:
    axis.bar(
        comparison_means.index,
        comparison_means.values,
        color=[COURSE_PALETTE[0], COURSE_PALETTE[1]],
    )
    axis.set_ylabel("Accuracy (%)")
axes[0].set(title="Bars from zero", ylim=(0, 100))
axes[1].set(title="Truncated bars exaggerate", ylim=(80, 85))
figure.suptitle("Same values, radically different visual impression")

assert np.ptp(comparison_means.to_numpy()) == 2.0
display(figure)
plt.close(figure)

Accuracy itself may be a poor metric under class imbalance or asymmetric cost. Fixing the axis does
not fix the scientific question.

## 16. Simulation studio: random walks need ensembles

A single simulated path is anecdotal. An ensemble shows variability. For independent steps
\(X_i \in \{-1,+1\}\), position after \(t\) steps is \(S_t=\sum_{i=1}^{t}X_i\), with expected value
zero and standard deviation approximately \(\sqrt{t}\).

In [ ]:
walk_rng = np.random.default_rng(seed=577)
walk_count = 500
step_count = 250
random_steps = walk_rng.choice(
    np.array([-1, 1], dtype=np.int8),
    size=(walk_count, step_count),
)
walk_positions = np.column_stack(
    [np.zeros(walk_count, dtype=np.int64), random_steps.cumsum(axis=1)]
)
walk_times = np.arange(step_count + 1)

assert walk_positions.shape == (500, 251)
assert np.all(walk_positions[:, 0] == 0)

In [ ]:
walk_median = np.median(walk_positions, axis=0)
walk_low, walk_high = np.quantile(walk_positions, [0.05, 0.95], axis=0)

figure, axes = plt.subplots(1, 2, figsize=(11, 4.5), layout="constrained")
for walk in walk_positions[:20]:
    axes[0].plot(walk_times, walk, alpha=0.35, linewidth=0.8)
axes[0].set(
    title="Twenty individual random walks",
    xlabel="Step",
    ylabel="Position",
)

axes[1].fill_between(
    walk_times,
    walk_low,
    walk_high,
    alpha=0.25,
    label="5th–95th percentile",
)
axes[1].plot(walk_times, walk_median, color="black", label="median")
axes[1].plot(
    walk_times,
    np.sqrt(walk_times),
    linestyle="--",
    label=r"$+\sqrt{t}$ reference",
)
axes[1].plot(walk_times, -np.sqrt(walk_times), linestyle="--")
axes[1].set(
    title="Ensemble distribution over time",
    xlabel="Step",
    ylabel="Position",
)
axes[1].legend(frameon=False)

display(figure)
plt.close(figure)

The percentile ribbon describes simulated positions across paths at each step. It is not a confidence
interval for an estimated mean, and values across time are strongly dependent within each path.

## 17. Simulation studio: Monte Carlo convergence

Sample points uniformly from the square `[-1, 1] × [-1, 1]`. The fraction inside the unit circle
estimates area ratio \(\pi/4\), so \(\hat\pi_n = 4k/n\). A convergence plot should show both error and
sample size; early volatility is expected.

In [ ]:
monte_carlo_rng = np.random.default_rng(seed=438)
point_count = 20_000
square_points = monte_carlo_rng.uniform(-1.0, 1.0, size=(point_count, 2))
inside_circle = np.square(square_points).sum(axis=1) <= 1.0
sample_sizes = np.arange(1, point_count + 1)
pi_estimates = 4.0 * np.cumsum(inside_circle) / sample_sizes

assert square_points.shape == (20_000, 2)
assert abs(pi_estimates[-1] - np.pi) < 0.05

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4.5), layout="constrained")
display_count = 2_000
axes[0].scatter(
    square_points[:display_count, 0],
    square_points[:display_count, 1],
    c=np.where(
        inside_circle[:display_count, np.newaxis],
        np.asarray(COURSE_PALETTE[0]),
        np.asarray(COURSE_PALETTE[3]),
    ),
    s=8,
    alpha=0.55,
    edgecolors="none",
)
axes[0].set(
    title="First 2,000 samples",
    xlabel="x",
    ylabel="y",
    aspect="equal",
)

axes[1].semilogx(sample_sizes, pi_estimates, linewidth=1)
axes[1].axhline(np.pi, color="black", linestyle="--", label=r"true $\pi$")
axes[1].set(
    title="Monte Carlo estimate stabilizes slowly",
    xlabel="Cumulative samples (log scale)",
    ylabel=r"Estimate of $\pi$",
)
axes[1].legend(frameon=False)

display(figure)
plt.close(figure)

Repeated independent runs are needed to estimate Monte Carlo variability. One seeded trajectory
demonstrates reproducibility, not reliability. Also notice that visualizing only 2,000 points prevents
overplotting while the numerical estimate uses all 20,000; the caption must disclose that difference.

## 18. Simulation studio: the central limit effect

The source distribution below is exponential and strongly right-skewed. We repeatedly sample means.
As sample size increases, the sampling distribution becomes more concentrated and more nearly
bell-shaped under the assumptions of the central limit theorem.

In [ ]:
clt_rng = np.random.default_rng(seed=577)
sample_sizes_for_clt = [1, 5, 30, 100]
sampling_records: list[dict[str, float | int]] = []

for sample_size in sample_sizes_for_clt:
    sample_means = clt_rng.exponential(
        scale=2.0,
        size=(4_000, sample_size),
    ).mean(axis=1)
    sampling_records.extend(
        {"sample_size": sample_size, "sample_mean": float(value)}
        for value in sample_means
    )

sampling_distributions = pd.DataFrame(sampling_records)

clt_grid = sns.displot(
    data=sampling_distributions,
    x="sample_mean",
    col="sample_size",
    col_wrap=2,
    bins=45,
    stat="density",
    common_bins=True,
    facet_kws={"sharex": True, "sharey": False},
    height=3,
)
clt_grid.set_axis_labels("Sample mean", "Density")
clt_grid.set_titles("n = {col_name}")
clt_grid.figure.suptitle("Sampling distributions from exponential data", y=1.02)

assert sampling_distributions.shape == (16_000, 2)
display(clt_grid.figure)
plt.close(clt_grid.figure)

The figure is not a proof, and convergence can be slow for heavy-tailed distributions. Shared x-scales
support spread comparison; independent y-scales avoid a nearly invisible tall density for large `n`.
That scale choice must be stated.

## 19. Simulation studio: nonlinear dynamics

The logistic map

\[
x_{t+1}=r x_t(1-x_t), \qquad 0 < x_0 < 1
\]

is a simple recurrence with fixed, periodic, and chaotic regimes. It is useful for distinguishing a
trajectory plot from a parameter-space summary.

In [ ]:
def logistic_trajectory(
    growth_rate: float,
    *,
    initial_state: float = 0.2,
    steps: int = 100,
) -> np.ndarray:
    """Iterate the logistic map for one growth rate.

    Parameters
    ----------
    growth_rate : float
        Logistic-map parameter, normally between 0 and 4.
    initial_state : float, default=0.2
        Initial state strictly between zero and one.
    steps : int, default=100
        Positive number of transitions.

    Returns
    -------
    numpy.ndarray
        State sequence of length ``steps + 1`` including the initial state.

    Raises
    ------
    ValueError
        If a parameter lies outside the documented domain.
    """

    if not 0.0 <= growth_rate <= 4.0:
        raise ValueError("growth_rate must lie in [0, 4]")
    if not 0.0 < initial_state < 1.0:
        raise ValueError("initial_state must lie strictly between 0 and 1")
    if steps <= 0:
        raise ValueError("steps must be positive")

    states = np.empty(steps + 1, dtype=np.float64)
    states[0] = initial_state
    for step in range(steps):
        states[step + 1] = growth_rate * states[step] * (1.0 - states[step])
    return states


figure, axes = plt.subplots(2, 2, figsize=(10, 6.5), sharex=True, layout="constrained")
for axis, growth_rate in zip(axes.flat, [2.8, 3.2, 3.5, 3.9], strict=True):
    trajectory = logistic_trajectory(growth_rate, steps=100)
    axis.plot(np.arange(101), trajectory, linewidth=1)
    axis.set(title=f"r = {growth_rate}", ylabel="State x")
for axis in axes[-1, :]:
    axis.set_xlabel("Iteration")
figure.suptitle("Logistic-map regimes")

display(figure)
plt.close(figure)

### A bifurcation diagram summarizes long-run states across parameters

Discarding a transient and plotting the last states reveals qualitative changes. The result depends
on parameter resolution, initial state, transient length, and retained iterations.

In [ ]:
growth_rates = np.linspace(2.5, 4.0, num=1_600)
states = np.full(growth_rates.shape, 0.2)

for _ in range(800):
    states = growth_rates * states * (1.0 - states)

bifurcation_rates: list[np.ndarray] = []
bifurcation_states: list[np.ndarray] = []
for _ in range(120):
    states = growth_rates * states * (1.0 - states)
    bifurcation_rates.append(growth_rates.copy())
    bifurcation_states.append(states.copy())

figure, axis = plt.subplots(figsize=(10, 5), layout="constrained")
axis.plot(
    np.concatenate(bifurcation_rates),
    np.concatenate(bifurcation_states),
    ",",
    color="black",
    alpha=0.2,
)
axis.set(
    title="Logistic-map bifurcation diagram",
    xlabel="Growth parameter r",
    ylabel="Long-run state x",
    xlim=(2.5, 4.0),
    ylim=(0.0, 1.0),
)

assert len(bifurcation_rates) == 120
display(figure)
plt.close(figure)

Chaos does not mean “random,” and a dense plot is not evidence of randomness. The recurrence is fully
deterministic. Numerical precision and finite iteration also affect the rendered structure.

## 20. Plotly adds interaction through a figure specification

Plotly Express maps DataFrame columns to visual properties and returns a graph-object `Figure`.
Hover, zoom, pan, legend toggling, facets, and animation can support exploration. Interactivity is
less suitable when the audience needs a stable printed comparison, accessible static alternative,
or reviewable default state.

In [ ]:
interactive_profile = (
    heat_data.groupby(
        ["material", "sensor_id", "distance_cm", "time_s"],
        observed=True,
    )["temperature_c"]
    .mean()
    .rename("mean_temperature_c")
    .reset_index()
)

interactive_figure = px.line(
    interactive_profile,
    x="time_s",
    y="mean_temperature_c",
    color="material",
    line_dash="material",
    facet_col="sensor_id",
    facet_col_wrap=2,
    markers=True,
    hover_data={
        "distance_cm": ":.1f",
        "mean_temperature_c": ":.2f",
    },
    labels={
        "time_s": "Time (s)",
        "mean_temperature_c": "Mean temperature (°C)",
        "material": "Material",
        "distance_cm": "Distance (cm)",
    },
    title="Interactive cooling profiles",
)
interactive_figure.update_layout(legend_title_text="Material")

assert len(interactive_figure.data) == 8
assert interactive_figure.layout.title.text == "Interactive cooling profiles"
interactive_figure

The object contains data traces and layout metadata that serialize to JSON and render with Plotly.js.
Inspect the figure contract rather than screenshot pixels when a structural test is sufficient.

### Interactive distributions and hover details

Interactivity can reveal exact values without labeling every mark. It cannot rescue poor aggregation,
missing context, or an inaccessible encoding.

In [ ]:
interactive_residuals = px.scatter(
    diagnostic_data.sample(400, random_state=438),
    x="expected_temperature_c",
    y="residual_c",
    color="material",
    symbol="sensor_id",
    facet_col="material",
    hover_data=["time_s", "sensor_id", "replicate"],
    opacity=0.65,
    labels={
        "expected_temperature_c": "Expected temperature (°C)",
        "residual_c": "Residual (°C)",
    },
    title="Interactive residual diagnostics",
)
interactive_residuals.add_hline(y=0.0, line_dash="dash", line_color="black")

assert len(interactive_residuals.data) == 8
interactive_residuals

## 21. Export is an engineering contract

Choose output by consumer:

- **SVG/PDF:** vector paths and text, strong for line art and publication workflows;
- **PNG:** fixed raster pixels, useful for slides, reports, and dense rasterized marks;
- **HTML:** interactive Plotly figure plus JavaScript, useful for browser exploration;
- **figure object/specification:** editable inside a Python workflow.

Record data/code versions, dimensions, units, and important plotting parameters. Do not overwrite a
reviewed result with an untraceable rerun.

In [ ]:
with TemporaryDirectory() as temporary_directory:
    output_directory = Path(temporary_directory)

    export_figure, export_axis = plt.subplots(
        figsize=(7.5, 4.5),
        layout="constrained",
    )
    export_axis.plot(copper_s1_means.index, copper_s1_means, marker="o")
    export_axis.set(
        title="Copper cooling at sensor S1",
        xlabel="Time (s)",
        ylabel="Mean temperature (°C)",
    )

    png_path = output_directory / "copper-cooling.png"
    svg_path = output_directory / "copper-cooling.svg"
    pdf_path = output_directory / "copper-cooling.pdf"
    export_figure.savefig(png_path, dpi=180, bbox_inches="tight")
    export_figure.savefig(svg_path, bbox_inches="tight")
    export_figure.savefig(pdf_path, bbox_inches="tight")
    plt.close(export_figure)

    html_path = output_directory / "interactive-cooling.html"
    interactive_figure.write_html(html_path, include_plotlyjs=True)

    assert png_path.read_bytes().startswith(b"\x89PNG")
    assert "<svg" in svg_path.read_text(encoding="utf-8")
    assert pdf_path.read_bytes().startswith(b"%PDF")
    assert html_path.stat().st_size > 1_000_000

print("Static and interactive exports verified in a temporary directory.")

Self-contained Plotly HTML is large because it embeds Plotly.js and works offline. Loading JavaScript
from a CDN produces smaller files but adds a network dependency. That is a deployment decision, not
a cosmetic setting.

## 22. Reusable plotting functions return objects

A package-quality plotting function should accept data and optionally an existing `Axes`, validate
the columns it owns, avoid hidden global state, label quantities and units, and return the objects a
caller may extend or test.

In [ ]:
def plot_temperature_profiles(
    data: pd.DataFrame,
    *,
    axis: Axes | None = None,
) -> tuple[Figure, Axes]:
    """Plot one mean temperature profile per material.

    Parameters
    ----------
    data : pandas.DataFrame
        Tidy observations containing ``material``, ``time_s``, and
        ``temperature_c``. Times are seconds and temperatures are degrees
        Celsius.
    axis : matplotlib.axes.Axes, optional
        Existing plotting region. A new figure and axes are created when
        omitted.

    Returns
    -------
    figure : matplotlib.figure.Figure
        Figure owning the returned axes.
    axis : matplotlib.axes.Axes
        Axes containing one line per observed material.

    Raises
    ------
    TypeError
        If ``data`` is not a pandas DataFrame.
    ValueError
        If required columns are absent or contain no valid observations.
    """

    if not isinstance(data, pd.DataFrame):
        raise TypeError("data must be a pandas.DataFrame")

    required_columns = {"material", "time_s", "temperature_c"}
    missing_columns = required_columns - set(data.columns)
    if missing_columns:
        missing_text = ", ".join(sorted(missing_columns))
        raise ValueError(f"data is missing required columns: {missing_text}")

    plot_data = data.dropna(subset=list(required_columns))
    if plot_data.empty:
        raise ValueError("data must contain at least one valid observation")

    if axis is None:
        figure, axis = plt.subplots(figsize=(8, 4.5), layout="constrained")
    else:
        figure = axis.figure

    summary = (
        plot_data.groupby(["material", "time_s"], observed=True)[
            "temperature_c"
        ]
        .mean()
        .rename("mean_temperature_c")
        .reset_index()
    )
    for material, group in summary.groupby("material", observed=True):
        axis.plot(
            group["time_s"],
            group["mean_temperature_c"],
            marker="o",
            label=str(material).title(),
        )

    axis.set(
        title="Mean temperature profiles",
        xlabel="Time (s)",
        ylabel="Mean temperature (°C)",
    )
    axis.legend(title="Material", frameon=False)
    return figure, axis


function_figure, function_axis = plot_temperature_profiles(
    heat_data.loc[heat_data["sensor_id"] == "S3"]
)

assert len(function_axis.lines) == 2
assert function_axis.get_xlabel() == "Time (s)"
assert function_axis.get_ylabel() == "Mean temperature (°C)"
display(function_figure)
plt.close(function_figure)

### Test semantics before pixels

Stable tests can check validation, number of semantic series, labels, axis scales, data attached to
artists, and successful export. Pixel snapshots can detect rendering changes but are sensitive to
fonts, backends, library versions, operating systems, and antialiasing. Use them only when visual
appearance is itself the contract and the rendering environment is controlled.

In [ ]:
test_data = pd.DataFrame(
    {
        "material": ["A", "A", "B", "B"],
        "time_s": [0, 1, 0, 1],
        "temperature_c": [10.0, 8.0, 9.0, 7.5],
    }
)
test_figure, test_axis = plot_temperature_profiles(test_data)

assert [line.get_label() for line in test_axis.lines] == ["A", "B"]
assert np.array_equal(test_axis.lines[0].get_xdata(), np.array([0, 1]))
assert np.allclose(test_axis.lines[0].get_ydata(), np.array([10.0, 8.0]))
assert test_axis.get_title() == "Mean temperature profiles"
plt.close(test_figure)

try:
    plot_temperature_profiles(pd.DataFrame({"time_s": [0]}))
except ValueError as error:
    assert "material" in str(error)
else:
    raise AssertionError("missing columns should be rejected")

## 23. Style state, reproducibility, and performance

Matplotlib style is global process state unless contained. Prefer `plt.rc_context` for temporary
changes inside reusable code. Close figures in loops and tests. Large scatters may require
rasterization, binning, aggregation, or purposefully sampled points.

In [ ]:
default_title_size = mpl.rcParams["axes.titlesize"]

with plt.rc_context({"axes.titlesize": 16, "axes.labelsize": 12}):
    context_figure, context_axis = plt.subplots(layout="constrained")
    context_axis.plot([0, 1], [0, 1])
    context_axis.set(title="Temporary style context", xlabel="Input", ylabel="Output")
    assert mpl.rcParams["axes.titlesize"] == 16
    plt.close(context_figure)

assert mpl.rcParams["axes.titlesize"] == default_title_size

For millions of observations, ask what visual resolution and question require. A 900-pixel-wide plot
cannot show ten million independent x positions. Compute honest aggregates, preserve rare groups,
and retain the full numerical analysis even when the display is downsampled. Libraries such as
Datashader specialize in large-data rasterization.

## 24. Choosing a library

| Need | Good starting point | Why |
| --- | --- | --- |
| exact static composition or publication figure | Matplotlib | explicit artist and layout control |
| rapid statistical exploration of tidy data | Seaborn | semantic mappings and statistical plot families |
| quick table-backed static plot | pandas `.plot` | convenient Matplotlib front end |
| hover, zoom, filtering, browser sharing | Plotly | interactive figure specification and HTML export |
| declarative grammar and JSON specification | Altair/Vega-Lite | concise encodings and transformations |
| linked web application or streaming interaction | Bokeh, Plotly Dash, Panel | browser-oriented application models |
| geographic vector data | GeoPandas plus Matplotlib/interactive map tool | geometry-aware tabular operations |
| extremely large point clouds | Datashader | aggregation directly to display pixels |

Do not add a dependency only because its defaults look attractive. Consider accessibility, artifact
format, ecosystem, deployment, maintenance, and whether the team can review the result.

## 25. Debugging a figure

1. restate the exact question and analytical unit;
2. inspect the plotted table's shape, dtypes, keys, missingness, and ranges;
3. identify every filter, group, aggregation, bin, and smoother;
4. verify x/y variables, units, scales, and limits;
5. check whether categories disappeared or reordered;
6. compare plotted point/line counts with expected groups;
7. isolate one panel and one group;
8. inspect the `Figure`, `Axes`, and artist data;
9. save through the same backend and dimensions as the final artifact; and
10. write a regression test for the failed semantic contract.

### Common failure modes

| Symptom | Likely cause | First check |
| --- | --- | --- |
| empty plot | filter removed rows or values are missing | plotted frame shape and `isna` |
| extra lines | hidden grouping semantic | unique hue/style/unit combinations |
| lines zigzag | x values unsorted within group | sort keys and sequence meaning |
| legend disagrees | layers use inconsistent mappings | labels on each artist/layer |
| unreadable points | overplotting | opacity, binning, sampling, facets |
| misleading smooth | bandwidth or model assumption | raw observations and sensitivity |
| wrong color emphasis | inappropriate palette/normalization | variable type and meaningful center |
| clipped labels | final dimensions/layout differ | export at target size |
| notebook memory grows | figures never closed | `plt.get_fignums()` |
| CI differs from laptop | backend/font/version differences | locked environment and semantic tests |
| interactive file fails offline | CDN JavaScript dependency | HTML export option |

## Guided practice: critique before rewriting

Consider a dashboard showing average model accuracy as bars from 91% to 92%, with a y-axis beginning
at 90%, no sample counts, no subgroup results, no uncertainty, and a rainbow palette.

1. List the analytical questions the chart cannot answer.
2. Identify which choices are misleading because of the bar encoding.
3. Propose one exploratory and one explanatory replacement.
4. State the denominator, comparison baseline, uncertainty, and accessibility requirements.

**Success criterion:** the redesign begins from a decision and failure cost, not a preferred chart
type, and it does not claim that a visually larger difference is practically important.

## Guided practice: compare distributions honestly

Using `diagnostic_data`, compare residuals across all four sensors and both materials.

1. Start with raw observations or ECDFs.
2. Add one compact summary without hiding `n`.
3. Use facets or redundant encodings accessible in grayscale.
4. Hold scales constant for the primary comparison.
5. Write a two-sentence text alternative.

**Success criterion:** the viewer can distinguish location, spread, sample size, and unusual values,
and the caption defines residual and any smoothing or whisker convention.

In [ ]:
# A scaffold: replace the empty axes with your distribution design.
practice_figure, practice_axes = plt.subplots(
    2,
    2,
    figsize=(10, 7),
    sharex=True,
    sharey=True,
    layout="constrained",
)
for practice_axis, sensor_id in zip(
    practice_axes.flat,
    ["S1", "S2", "S3", "S4"],
    strict=True,
):
    sensor_residuals = diagnostic_data.loc[
        diagnostic_data["sensor_id"] == sensor_id
    ]
    sns.ecdfplot(
        data=sensor_residuals,
        x="residual_c",
        hue="material",
        ax=practice_axis,
    )
    practice_axis.set_title(f"Sensor {sensor_id}")

assert all(len(axis.lines) == 2 for axis in practice_axes.flat)
display(practice_figure)
plt.close(practice_figure)

## Independent practice: build and test a calibration plot

Simulate binary outcomes with predicted scores, then write
`plot_calibration(data: pd.DataFrame, *, bins: int, axis: Axes | None = None)`.

Requirements:

- validate scores in `[0, 1]`, Boolean/0–1 outcomes, positive bin count, and required columns;
- display mean predicted score versus observed event frequency per nonempty bin;
- show bin counts and the `y=x` reference;
- label both axes as proportions, not “probability” unless calibration is justified;
- return figure, axes, and the computed calibration table; and
- test the computed table separately from the artists.

**Success criterion:** document empty-bin behavior, uncertainty limitations, and why a diagonal-looking
plot alone does not prove calibration on future or subgroup data.

## Independent practice: tell two stories with one simulation

Use the random-walk ensemble to create:

1. an exploratory diagnostic for a scientist checking the simulation; and
2. an explanatory figure for a general audience learning why typical displacement grows with time.

Keep the numerical simulation identical. Change only transformation, layout, annotation, and detail.

**Success criterion:** explain what each figure removes, what it emphasizes, and why neither implies
that a single path will remain inside the displayed percentile band.

## Extension: animation and interaction require a temporal reason

Animation is useful when change, transition, or process order is the subject. It is poor when viewers
must compare distant frames from memory. Consider small multiples or a trajectory with time encoded
directly before adding motion.

For Matplotlib, `matplotlib.animation.FuncAnimation` produces frame sequences. For Plotly, an
`animation_frame` column can generate controls. Before shipping either, decide replay controls,
reduced-motion accessibility, frame rate, artifact size, and a static alternative.

**Success criterion:** justify what temporal relationship cannot be communicated as clearly in a
static figure.

## Extension: visual regression strategy

Design a test portfolio for a publication figure:

- unit tests for transformations and uncertainty calculations;
- schema tests for plotted data;
- semantic artist tests for labels, groups, scales, and references;
- export smoke tests for PNG, SVG, and PDF;
- a tightly controlled image comparison only if pixel layout is contractual; and
- human review for interpretation, accessibility, and scientific honesty.

**Success criterion:** identify what each layer can detect and which defects still require expert
review.

## Retrieval practice

Answer without executing code:

1. Distinguish a Matplotlib `Figure`, `Axes`, `Axis`, and `Artist`.
2. What is the difference between a mark and a visual encoding?
3. When does connecting points with a line make a semantic claim?
4. Why should raw observations often precede an aggregate?
5. Distinguish standard deviation, standard error, and a confidence interval.
6. How do histogram bins and KDE bandwidth change visible structure?
7. Why can a truncated y-axis be especially misleading for bars?
8. When are shared facet scales necessary?
9. What does a Plotly figure contain, and what does HTML export add?
10. Which parts of a plotting function should be tested numerically rather than visually?
11. Why is a fixed random seed insufficient evidence of simulation robustness?
12. Name one important conclusion each scientific studio cannot establish.

## Takeaway

```text
question → trustworthy data → explicit transformation
         → appropriate marks and encodings → honest scales and uncertainty
         → accessible annotation → reproducible artifact → review
```

Matplotlib gives precise ownership of figures and artists. Seaborn accelerates statistical graphics
over tidy data. pandas offers convenience, and Plotly adds interactive exploration. None chooses the
scientific comparison, uncertainty model, or ethical framing for you.

The next notebook returns to deeper NumPy indexing, reshaping, broadcasting, and vectorized numerical
computation—the machinery beneath many of these simulations.

## Further reading

- [Matplotlib quick start](https://matplotlib.org/stable/users/explain/quick_start.html)
- [Matplotlib: choosing colormaps](https://matplotlib.org/stable/users/explain/colors/colormaps.html)
- [Matplotlib backends](https://matplotlib.org/stable/users/explain/figure/backends.html)
- [Seaborn introduction](https://seaborn.pydata.org/tutorial/introduction.html)
- [Seaborn function overview](https://seaborn.pydata.org/tutorial/function_overview.html)
- [Seaborn relational plots](https://seaborn.pydata.org/tutorial/relational.html)
- [Seaborn regression plots](https://seaborn.pydata.org/tutorial/regression.html)
- [Plotly Express](https://plotly.com/python/plotly-express/)
- [Plotly graph objects](https://plotly.com/python/graph-objects/)
- [Plotly interactive HTML export](https://plotly.com/python/interactive-html-export/)
- [Python `pathlib`](https://docs.python.org/3/library/pathlib.html)